### 1.问题
智能体工作越久，messages数组越胖。  
每次读文件、跑命令的输出都永久留在上下文里。
### 2.解决方法
1. 父智能体有一个task工具。子智能体拥有除了task以外的所有基础工具（禁止递归生成）。
```python
PARENT_TOOLS = CHILD_TOOLS + [
    {
        "type": "function",
        "function": {
            "name": "task",
            "description": "Spawn a subagent with fresh context."
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                    },
                    "required": ["prompt"],
                }
            }
        }
    }
]
```
2. 子智能体以message=[]启动，运行自己的循环。只有最终文本返回给父智能体。
```python
import json

def run_subagent(prompt: str) -> str:
    sub_messages = [
        {"role": "system", "content": SUBAGENT_SYSTEM},
        {"role": "user", "content": prompt},
    ]
    response = None

    for _ in range(30):
        response = client.chat.completions.create(
            model=MODEL,
            messages=sub_messages,
            tools=CHILD_TOOLS,
            max_tokens=8000,
        )
        message = response.choices[0].message
        sub_messages.append(message.model_dump())

        if not message.tool_calls:
            break

        for tool_call in message.tool_calls:
            handler = TOOL_HANDLERS.get(tool_call.function.name)
            output = (
                handler(**json.loads(tool_call.function.arguments))
                if handler
                else f"Unknown tool: {tool_call.function.name}"
            )
            sub_messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output)[:50000],
            })
    else:
        response = client.chat.completions.create(
            model=MODEL,
            messages=sub_messages,
            tools=CHILD_TOOLS,
            max_tokens=8000,
        )

    return response.choices[0].message.content or "(no summary)"
```
子智能体可能跑了30+次工具调用，但整个消息历史直接丢弃。父智能体收到的只是一段摘要文本，作为普通的tool_result返回。

In [4]:
#!/usr/bin/env python3
# Harness: context isolation -- protecting the model's clarity of thought.
"""
s04_subagent_chat_completions.py - Subagents (OpenAI-compatible Chat Completions)

Spawn a child agent with fresh messages=[]. The child works in its own
context, sharing the filesystem, then returns only a summary to the parent.
"""

import json
import os
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

WORKDIR = Path.cwd()
MODEL = "Pro/MiniMaxAI/MiniMax-M2.5"

# Optional: support custom compatible endpoint
openai_base_url = os.getenv("OPENAI_BASE_URL")
client = OpenAI(base_url=openai_base_url) if openai_base_url else OpenAI()

SYSTEM = f"You are a coding agent at {WORKDIR}. Use the task tool to delegate exploration or subtasks."
SUBAGENT_SYSTEM = f"You are a coding subagent at {WORKDIR}. Complete the given task, then summarize your findings."


# -- Tool implementations shared by parent and child --
def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path


def run_bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(
            command,
            shell=True,
            cwd=WORKDIR,
            capture_output=True,
            text=True,
            timeout=120,
        )
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"


def run_read(path: str, limit: int = None) -> str:
    try:
        lines = safe_path(path).read_text().splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"... ({len(lines) - limit} more)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"


def run_write(path: str, content: str) -> str:
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content)
        return f"Wrote {len(content)} bytes"
    except Exception as e:
        return f"Error: {e}"


def run_edit(path: str, old_text: str, new_text: str) -> str:
    try:
        fp = safe_path(path)
        content = fp.read_text()
        if old_text not in content:
            return f"Error: Text not found in {path}"
        fp.write_text(content.replace(old_text, new_text, 1))
        return f"Edited {path}"
    except Exception as e:
        return f"Error: {e}"


TOOL_HANDLERS = {
    "bash":       lambda **kw: run_bash(kw["command"]),
    "read_file":  lambda **kw: run_read(kw["path"], kw.get("limit")),
    "write_file": lambda **kw: run_write(kw["path"], kw["content"]),
    "edit_file":  lambda **kw: run_edit(kw["path"], kw["old_text"], kw["new_text"]),
}


def _function_tool(name: str, description: str, parameters: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": parameters,
        },
    }


# Child gets all base tools except task (no recursive spawning)
CHILD_TOOLS = [
    _function_tool(
        "bash",
        "Run a shell command.",
        {
            "type": "object",
            "properties": {
                "command": {"type": "string"}
            },
            "required": ["command"],
            "additionalProperties": False,
        },
    ),
    _function_tool(
        "read_file",
        "Read file contents.",
        {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "limit": {"type": "integer"},
            },
            "required": ["path"],
            "additionalProperties": False,
        },
    ),
    _function_tool(
        "write_file",
        "Write content to file.",
        {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "content": {"type": "string"},
            },
            "required": ["path", "content"],
            "additionalProperties": False,
        },
    ),
    _function_tool(
        "edit_file",
        "Replace exact text in file.",
        {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "old_text": {"type": "string"},
                "new_text": {"type": "string"},
            },
            "required": ["path", "old_text", "new_text"],
            "additionalProperties": False,
        },
    ),
]


def _message_text(msg) -> str:
    return msg.content or ""


# -- Subagent: fresh context, filtered tools, summary-only return --
def run_subagent(prompt: str) -> str:
    sub_messages = [
        {"role": "system", "content": SUBAGENT_SYSTEM},
        {"role": "user", "content": prompt},
    ]  # fresh context

    for _ in range(30):  # safety limit
        response = client.chat.completions.create(
            model=MODEL,
            messages=sub_messages,
            tools=CHILD_TOOLS,
            tool_choice="auto",
            max_completion_tokens=8000,
        )

        msg = response.choices[0].message

        assistant_message = {
            "role": "assistant",
            "content": msg.content or "",
        }
        if msg.tool_calls:
            assistant_message["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": tc.type,
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in msg.tool_calls
            ]

        sub_messages.append(assistant_message)

        if not msg.tool_calls:
            break

        for tc in msg.tool_calls:
            handler = TOOL_HANDLERS.get(tc.function.name)
            args = json.loads(tc.function.arguments) if tc.function.arguments else {}
            output = handler(**args) if handler else f"Unknown tool: {tc.function.name}"

            sub_messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(output)[:50000],
            })

    # Only the final text returns to the parent -- child context is discarded
    return _message_text(msg) or "(no summary)"


# -- Parent tools: base tools + task dispatcher --
PARENT_TOOLS = CHILD_TOOLS + [
    _function_tool(
        "task",
        "Spawn a subagent with fresh context. It shares the filesystem but not conversation history.",
        {
            "type": "object",
            "properties": {
                "prompt": {"type": "string"},
                "description": {
                    "type": "string",
                    "description": "Short description of the task"
                },
            },
            "required": ["prompt"],
            "additionalProperties": False,
        },
    ),
]


def agent_loop(messages: list) -> str:
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=PARENT_TOOLS,
            tool_choice="auto",
            max_completion_tokens=8000,
        )

        msg = response.choices[0].message

        assistant_message = {
            "role": "assistant",
            "content": msg.content or "",
        }
        if msg.tool_calls:
            assistant_message["tool_calls"] = [
                {
                    "id": tc.id,
                    "type": tc.type,
                    "function": {
                        "name": tc.function.name,
                        "arguments": tc.function.arguments,
                    },
                }
                for tc in msg.tool_calls
            ]

        messages.append(assistant_message)

        if not msg.tool_calls:
            return _message_text(msg)

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments) if tc.function.arguments else {}

            if tc.function.name == "task":
                desc = args.get("description", "subtask")
                print(f"> task ({desc}): {args['prompt'][:80]}")
                output = run_subagent(args["prompt"])
            else:
                handler = TOOL_HANDLERS.get(tc.function.name)
                output = handler(**args) if handler else f"Unknown tool: {tc.function.name}"

            print(f"  {str(output)[:200]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(output),
            })


if __name__ == "__main__":
    history = [{"role": "system", "content": SYSTEM}]
    while True:
        try:
            query = input("请输入: ")
        except (EOFError, KeyboardInterrupt):
            break

        if query.strip().lower() in ("q", "exit", ""):
            break

        history.append({"role": "user", "content": query})
        final_text = agent_loop(history)

        if final_text:
            print(final_text)
        print()



你好！有什么我可以帮助你的吗？我是一个编码助手，可以帮助你处理文件、编写代码、运行命令等任务。请问有什么需要我帮忙的吗？

> task (创建hello.txt文件): 请在当前目录下创建一个名为hello.txt的文件，并向其中写入内容"hello"。完成后确认文件已创建成功。
  

任务完成！我已成功完成以下操作：

1. **创建文件**：在当前目录下创建了名为 `hello.txt` 的文件
2. **写入内容**：向文件中写入了内容 "hello"
3. **确认成功**：读取文件内容，确实为 "hello"

文件已创建并验证成功。


任务完成！子智能体已经成功：

1. ✅ 在当前目录下创建了 `hello.txt` 文件
2. ✅ 向文件中写入了内容 "hello"

文件已创建并验证成功。还需要我帮你做其他事情吗？

